# Email Recruiter Pipeline
Pulls all emails from Yahoo INBOX → SQLite → filters recruiter domains → cold outreach tracker

In [16]:
# ── Imports ──────────────────────────────────────────────────────────────────
import imaplib
import email
import sqlite3
import pandas as pd
from email.utils import parseaddr, parsedate_to_datetime
from dotenv import find_dotenv, load_dotenv
import os

load_dotenv(find_dotenv())
print('Libraries loaded ✓')

Libraries loaded ✓


In [17]:
# ── Config ────────────────────────────────────────────────────────────────────
YAHOO_EMAIL = 'brandoncdedwards@yahoo.com'
YMAIL_PW    = os.getenv('ymail_password')   # loaded from .env — never printed
DB_PATH     = 'emails.db'

# Domains treated as personal / non-recruiter — extend as needed
PERSONAL_DOMAINS = {
    'yahoo.com', 'ymail.com',
    'gmail.com', 'googlemail.com',
    'hotmail.com', 'hotmail.co.uk',
    'outlook.com', 'live.com', 'msn.com',
    'aol.com',
    'icloud.com', 'me.com', 'mac.com',
    'protonmail.com', 'proton.me',
    'comcast.net', 'att.net', 'verizon.net',
    'sbcglobal.net', 'bellsouth.net',
    'zoho.com',
}

print(f'Config ready | DB: {DB_PATH}')

Config ready | DB: emails.db


In [18]:
# ── Create SQLite DB & Tables ─────────────────────────────────────────────────
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# raw_emails — every email pulled from IMAP (upsert-safe via UNIQUE on msg_id)
cur.execute('''
    CREATE TABLE IF NOT EXISTS raw_emails (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        msg_id      TEXT    UNIQUE,
        from_name   TEXT,
        from_email  TEXT,
        domain      TEXT,
        subject     TEXT,
        date_sent   TEXT
    )
''')

# recruiters — deduplicated, one row per email address, with outreach tracking
cur.execute('''
    CREATE TABLE IF NOT EXISTS recruiters (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        from_name       TEXT,
        from_email      TEXT    UNIQUE,
        domain          TEXT,
        email_count     INTEGER DEFAULT 1,
        first_contact   TEXT,
        last_contact    TEXT,
        outreach_sent   INTEGER DEFAULT 0,
        outreach_date   TEXT
    )
''')

con.commit()
print('Database & tables ready ✓')

Database & tables ready ✓


In [19]:
# ── Connect to Yahoo IMAP ─────────────────────────────────────────────────────
imap = imaplib.IMAP4_SSL('imap.mail.yahoo.com')
imap.login(YAHOO_EMAIL, YMAIL_PW)
print('IMAP connected ✓')

IMAP connected ✓


In [ ]:
# ── Fetch Headers in Batches (avoids Yahoo rate limiting) ────────────────────
# Fetches 150 emails per IMAP request instead of 1 — ~67x fewer API calls

imap.select('INBOX', readonly=True)
status, messages = imap.search(None, 'ALL')
email_ids    = messages[0].split()
total_emails = len(email_ids)
print(f'Emails found in INBOX: {total_emails}')

FETCH_CMD  = '(BODY.PEEK[HEADER.FIELDS (FROM DATE SUBJECT MESSAGE-ID)])'
BATCH_SIZE = 150   # tune down to 50 if still hitting limits
records    = []
errors     = 0

def parse_header_item(item, fallback_id=''):
    """Parse a single header blob into a record dict, or return None."""
    try:
        msg        = email.message_from_bytes(item)
        from_raw   = msg.get('From', '')
        name, addr = parseaddr(from_raw)
        addr       = addr.lower().strip()
        if not addr or '@' not in addr:
            return None
        domain  = addr.split('@')[-1]
        msg_id  = msg.get('Message-ID', f'no-id-{fallback_id}').strip()
        subject = msg.get('Subject', '').strip()
        try:
            date_sent = parsedate_to_datetime(msg.get('Date', '')).isoformat()
        except Exception:
            date_sent = None
        return {
            'msg_id':     msg_id,
            'from_name':  name.strip(),
            'from_email': addr,
            'domain':     domain,
            'subject':    subject,
            'date_sent':  date_sent,
        }
    except Exception:
        return None

# Process in batches
for batch_start in range(0, total_emails, BATCH_SIZE):
    batch   = email_ids[batch_start : batch_start + BATCH_SIZE]
    ids_str = b','.join(batch)          # e.g. b'1,2,3,...,150'

    try:
        status, data = imap.fetch(ids_str, FETCH_CMD)
        # Batch response is a flat list — tuples hold the header bytes, b')' are separators
        for item in data:
            if isinstance(item, tuple) and len(item) == 2:
                rec = parse_header_item(item[1], fallback_id=batch_start)
                if rec:
                    records.append(rec)
    except Exception as e:
        errors += len(batch)
        print(f'  Batch error at {batch_start}: {e}')

    processed = min(batch_start + BATCH_SIZE, total_emails)
    if processed % 1000 == 0 or processed == total_emails:
        print(f'  Fetched {processed} / {total_emails} ...')

imap.logout()
print(f'\nDone. Parsed {len(records)} emails | Errors: {errors}')

In [ ]:
# ── Insert into raw_emails (skip duplicates via IGNORE) ───────────────────────
cur.executemany('''
    INSERT OR IGNORE INTO raw_emails
        (msg_id, from_name, from_email, domain, subject, date_sent)
    VALUES
        (:msg_id, :from_name, :from_email, :domain, :subject, :date_sent)
''', records)

con.commit()
total = cur.execute('SELECT COUNT(*) FROM raw_emails').fetchone()[0]
print(f'raw_emails table: {total} total rows')

In [ ]:
# ── Build Recruiters Table ────────────────────────────────────────────────────
# Pull non-personal domains from raw_emails, deduplicate by from_email
df = pd.read_sql('SELECT * FROM raw_emails', con)

recruiter_df = (
    df[~df['domain'].isin(PERSONAL_DOMAINS)]
    .dropna(subset=['from_email'])
    .groupby('from_email')
    .agg(
        from_name    = ('from_name',  'first'),
        domain       = ('domain',     'first'),
        email_count  = ('msg_id',     'count'),
        first_contact = ('date_sent', 'min'),
        last_contact  = ('date_sent', 'max'),
    )
    .reset_index()
    .sort_values('last_contact', ascending=False)
)

print(f'Unique recruiter emails found: {len(recruiter_df)}')
recruiter_df.head(10)

In [ ]:
# ── Upsert into recruiters table ─────────────────────────────────────────────
for _, row in recruiter_df.iterrows():
    cur.execute('''
        INSERT INTO recruiters
            (from_name, from_email, domain, email_count, first_contact, last_contact)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT(from_email) DO UPDATE SET
            email_count   = excluded.email_count,
            last_contact  = excluded.last_contact,
            first_contact = excluded.first_contact
    ''', (
        row['from_name'], row['from_email'], row['domain'],
        row['email_count'], row['first_contact'], row['last_contact']
    ))

con.commit()
total = cur.execute('SELECT COUNT(*) FROM recruiters').fetchone()[0]
print(f'recruiters table: {total} rows')

In [ ]:
# ── Quick Stats ───────────────────────────────────────────────────────────────
print('=== Top 15 Recruiter Domains ===')
top_domains = pd.read_sql('''
    SELECT domain, COUNT(*) AS recruiter_count
    FROM recruiters
    GROUP BY domain
    ORDER BY recruiter_count DESC
    LIMIT 15
''', con)
print(top_domains.to_string(index=False))

print('\n=== Outreach Status ===')
status = pd.read_sql('''
    SELECT
        SUM(CASE WHEN outreach_sent = 0 THEN 1 ELSE 0 END) AS not_contacted,
        SUM(CASE WHEN outreach_sent = 1 THEN 1 ELSE 0 END) AS contacted
    FROM recruiters
''', con)
print(status.to_string(index=False))

In [ ]:
# ── Mark Outreach Sent ────────────────────────────────────────────────────────
# Run this after you send a cold email to mark it as contacted
# Example: mark a single address

# email_to_mark = 'recruiter@somecompany.com'
# cur.execute('''
#     UPDATE recruiters
#     SET outreach_sent = 1, outreach_date = DATE('now')
#     WHERE from_email = ?
# ''', (email_to_mark,))
# con.commit()
# print(f'Marked {email_to_mark} as contacted')

print('Outreach marking cell ready — uncomment and set email_to_mark to use')

In [ ]:
# ── Export to CSV (optional) ──────────────────────────────────────────────────
export_df = pd.read_sql('''
    SELECT from_name, from_email, domain, email_count,
           first_contact, last_contact, outreach_sent, outreach_date
    FROM recruiters
    WHERE outreach_sent = 0
    ORDER BY last_contact DESC
''', con)

export_df.to_csv('recruiters_to_contact.csv', index=False)
print(f'Exported {len(export_df)} recruiters to recruiters_to_contact.csv')

con.close()